# Bloch vs PINN-based Dispersion Comparison

Compare:
1. Analytical Bloch branches (monoatomic + mass-in-mass),
2. PINN pointwise ridge picking,
3. PINN continuity-tracked ridges.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

sys.path.append(str(Path('.').resolve()))

from pinn_dispersion_from_mat import (
    load_pinn_results,
    resample_to_uniform,
    compute_spectrum,
    extract_ridge,
    linear_dispersion,
)
from continuity_multi_ridge_tracker import TrackerConfig, run_tracker_from_mat
from dispersion_theory import monoatomic_dispersion, mass_in_mass_dispersion

In [ ]:
# ---------------- USER SETTINGS ----------------
mat_file = '../Result/pinn_data.mat'

# Use your structural parameters
k_coupling = 1.0
mx = 1.0
my = 0.3
k_int = 8.6483   # effective internal spring for Bloch mass-in-mass model

omega_min = 0.01
skip_transient = 0.20
n_harmonic = 4
pts_per_cycle = 12

tracker_cfg = TrackerConfig(
    n_ridges=3,
    omega_min=omega_min,
    skip_transient=skip_transient,
    n_harmonic=n_harmonic,
    pts_per_cycle=pts_per_cycle,
    max_candidates_per_k=6,
    peak_prominence_db=2.0,
    jump_penalty=1.5,
    suppression_half_width_bins=2,
)

In [ ]:
# PINN spectrum + pointwise ridge
t_raw, x_raw, params = load_pinn_results(mat_file)
X_raw = x_raw.T

omega_max_lin = linear_dispersion(np.pi, k_coupling, mx)
t_u, X_u, dt, omega_nyq = resample_to_uniform(
    t_raw, X_raw, omega_max_lin,
    n_harmonic=n_harmonic,
    pts_per_cycle=pts_per_cycle,
)

k_pos, omega_pos, spectrum = compute_spectrum(t_u, X_u, skip_transient=skip_transient)
k_pw, omega_pw = extract_ridge(k_pos, omega_pos, spectrum, omega_min=omega_min)

# Continuity-based ridges
cont_out = run_tracker_from_mat(mat_file, k_coupling=k_coupling, mx=mx, cfg=tracker_cfg)
k_cont = cont_out['k_pos']
ridges_cont = cont_out['ridges_omega']

# Bloch analytical curves
k_line = np.linspace(0, np.pi, 400)
omega_bloch_mono = monoatomic_dispersion(k_line, k_coupling, mx)
omega_bloch_lo, omega_bloch_hi = mass_in_mass_dispersion(k_line, k_coupling, mx, my, k_int)

print('Loaded:', mat_file)
print('params keys:', list(params.keys()))

In [ ]:
# Comparison plot
plt.figure(figsize=(10, 6.5))

# Bloch reference
plt.plot(k_line/np.pi, omega_bloch_mono, 'k:', lw=1.8, label='Bloch monoatomic')
plt.plot(k_line/np.pi, omega_bloch_lo, 'k--', lw=2.0, label='Bloch acoustic (mass-in-mass)')
plt.plot(k_line/np.pi, omega_bloch_hi, 'k-.', lw=2.0, label='Bloch optical (mass-in-mass)')

# Pointwise ridge
valid_pw = np.isfinite(omega_pw)
plt.plot(k_pw[valid_pw]/np.pi, omega_pw[valid_pw], color='tab:orange', lw=2.2, label='PINN pointwise ridge')

# Continuity-tracked ridges
for i, om in enumerate(ridges_cont, start=1):
    valid = np.isfinite(om)
    plt.plot(k_cont[valid]/np.pi, om[valid], lw=2.0, label=f'PINN continuity ridge {i}')

plt.xlabel(r'Wavenumber $k/\pi$')
plt.ylabel(r'Frequency $\omega$ (rad/s)')
plt.title('Bloch vs PINN-based dispersion (pointwise + continuity)')
plt.grid(alpha=0.25)
plt.legend(loc='best', fontsize=8, ncol=2)
plt.tight_layout()
plt.show()